# Day 2 — Stage 3: OCR (Tesseract + PaddleOCR)

**Goal:** run both OCR engines on preprocessed images, structure their 
outputs into a common format, and compare results.

**Common format per detected word:**
- `text` — the recognized string
- `conf` — confidence (0.0 to 1.0, normalized)
- `bbox` — bounding box as (x, y, width, height)

This lets downstream stages (layout analysis, field extraction) work 
with either engine's output without caring which one produced it.

In [1]:
import cv2
import sys
import numpy as np
import pytesseract
from pathlib import Path
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.preprocessing import preprocess

# Load and preprocess our test image
image_path = project_root / "sample_images" / "sroie_05.jpg"
image = cv2.imread(str(image_path))
preprocessed = preprocess(image)

print("Image loaded:", image_path.name)
print("Original shape:", image.shape)
print("Preprocessed shape:", preprocessed.shape)

Image loaded: sroie_05.jpg
Original shape: (1646, 544, 3)
Preprocessed shape: (1646, 544)


In [2]:
def run_tesseract(image):
    """
    Run Tesseract OCR and return results in common format.
    
    Args:
        image: numpy array (BGR or grayscale).
    
    Returns:
        List of dicts, each with keys: text, conf, bbox (x, y, w, h)
    """
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    
    results = []
    for i in range(len(data["text"])):
        text = data["text"][i].strip()
        conf = int(data["conf"][i])
        
        if conf < 0 or text == "":
            continue
        
        results.append({
            "text": text,
            "conf": conf / 100.0,  # normalize to 0.0-1.0
            "bbox": (data["left"][i], data["top"][i], 
                     data["width"][i], data["height"][i]),
        })
    
    return results


# Run on preprocessed image
tess_results = run_tesseract(preprocessed)

print(f"Tesseract detected {len(tess_results)} words")
print(f"\nFirst 10:")
for r in tess_results[:10]:
    x, y, w, h = r["bbox"]
    print(f"  conf={r['conf']:.2f}  bbox=({x:>4},{y:>4},{w:>3},{h:>3})  text='{r['text']}'")

Tesseract detected 150 words

First 10:
  conf=0.00  bbox=(  75, 181,133, 31)  text='Treple'
  conf=0.27  bbox=( 291, 173, 93, 65)  text='ted'
  conf=0.00  bbox=( 407, 175,102, 37)  text='LOc8,'
  conf=0.92  bbox=(  33, 285, 96, 26)  text='Guardian'
  conf=0.96  bbox=( 142, 285, 71, 25)  text='Health'
  conf=0.96  bbox=( 225, 285, 36, 25)  text='And'
  conf=0.62  bbox=( 274, 285, 70, 25)  text='Beauty'
  conf=0.81  bbox=( 360, 285, 33, 24)  text='Sdn'
  conf=0.84  bbox=( 408, 284, 34, 25)  text='Phd'
  conf=0.92  bbox=(  31, 316, 60, 27)  text='Jalan'


In [3]:
from paddleocr import PaddleOCR

# Initialize once — this downloads models on first run (already done Day 1)
# use_angle_cls=True enables text direction detection (handles upside-down text)
paddle = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)

# PaddleOCR expects a file path or RGB numpy array
# Pass the file path for simplest usage — it does its own preprocessing internally
paddle_raw = paddle.ocr(str(image_path), cls=True)

# PaddleOCR returns a nested list: [page][line] where each line is [bbox, (text, conf)]
# For a single image there's one page, so results are in paddle_raw[0]
print(f"PaddleOCR detected {len(paddle_raw[0])} text regions")
print(f"\nFirst 5:")
for item in paddle_raw[0][:5]:
    bbox_points = item[0]    # 4 corner points [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    text = item[1][0]
    conf = item[1][1]
    print(f"  conf={conf:.2f}  text='{text}'")
    print(f"           bbox corners: {[[int(c) for c in pt] for pt in bbox_points]}")

download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to C:\Users\dsbha/.paddleocr/whl\det\en\en_PP-OCRv3_det_infer\en_PP-OCRv3_det_infer.tar


100%|██████████| 4.00M/4.00M [00:18<00:00, 218kiB/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to C:\Users\dsbha/.paddleocr/whl\rec\en\en_PP-OCRv4_rec_infer\en_PP-OCRv4_rec_infer.tar


100%|██████████| 10.2M/10.2M [01:07<00:00, 153kiB/s] 


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to C:\Users\dsbha/.paddleocr/whl\cls\ch_ppocr_mobile_v2.0_cls_infer\ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2.19M/2.19M [00:25<00:00, 86.3kiB/s]


PaddleOCR detected 54 text regions

First 5:
  conf=0.64  text='78C7 pm e dn4S'
           bbox corners: [[76, 175], [506, 173], [507, 214], [77, 216]]
  conf=0.93  text='Guardian Health And Heauty Sdn Bhd'
           bbox corners: [[32, 283], [444, 283], [444, 312], [32, 312]]
  conf=0.95  text='Jalan Loke Yew Bentong'
           bbox corners: [[29, 314], [299, 317], [299, 346], [29, 343]]
  conf=0.94  text='90 Ground Floor'
           bbox corners: [[31, 350], [211, 350], [211, 379], [31, 379]]
  conf=0.96  text='Jalan Bentong'
           bbox corners: [[29, 382], [190, 382], [190, 412], [29, 412]]


In [4]:
def run_paddleocr(paddle_engine, image_path):
    """
    Run PaddleOCR and return results in common format.
    
    Args:
        paddle_engine: initialized PaddleOCR instance.
        image_path: path to image file (string or Path).
    
    Returns:
        List of dicts with keys: text, conf, bbox (x, y, w, h)
    """
    raw = paddle_engine.ocr(str(image_path), cls=True)
    
    results = []
    for item in raw[0]:
        points = item[0]          # [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
        text = item[1][0]
        conf = float(item[1][1])  # already 0.0-1.0
        
        # Convert 4-corner polygon to axis-aligned (x, y, w, h)
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        x = int(min(xs))
        y = int(min(ys))
        w = int(max(xs) - min(xs))
        h = int(max(ys) - min(ys))
        
        results.append({
            "text": text,
            "conf": conf,
            "bbox": (x, y, w, h),
        })
    
    return results


paddle_results = run_paddleocr(paddle, image_path)

print(f"PaddleOCR: {len(paddle_results)} regions")
print(f"\nFirst 10:")
for r in paddle_results[:10]:
    x, y, w, h = r["bbox"]
    print(f"  conf={r['conf']:.2f}  bbox=({x:>4},{y:>4},{w:>3},{h:>3})  text='{r['text']}'")

PaddleOCR: 54 regions

First 10:
  conf=0.64  bbox=(  76, 173,431, 43)  text='78C7 pm e dn4S'
  conf=0.93  bbox=(  32, 283,412, 29)  text='Guardian Health And Heauty Sdn Bhd'
  conf=0.95  bbox=(  29, 314,270, 32)  text='Jalan Loke Yew Bentong'
  conf=0.94  bbox=(  31, 350,180, 29)  text='90 Ground Floor'
  conf=0.96  bbox=(  29, 382,161, 30)  text='Jalan Bentong'
  conf=0.94  bbox=(  31, 415,183, 29)  text='Tel:09-222 6498'
  conf=0.95  bbox=(  29, 448,269, 29)  text='Company Reg #1101083-T'
  conf=0.96  bbox=(  31, 482,256, 29)  text='GST Reg #000899874816'
  conf=0.93  bbox=( 389, 545, 34, 31)  text='RM'
  conf=0.94  bbox=(  31, 578,316, 32)  text='121093307G BXTISSU4X150'


In [5]:
print(f"{'Tesseract':<50} | {'PaddleOCR':<50}")
print("-" * 103)

# PaddleOCR returns lines, Tesseract returns words.
# For comparison, show PaddleOCR lines alongside Tesseract words 
# that fall in the same vertical region (similar y coordinate).

for p in paddle_results:
    px, py, pw, ph = p["bbox"]
    p_mid_y = py + ph / 2
    
    # Find all Tesseract words whose vertical center is within this line's bbox
    matching_tess = []
    for t in tess_results:
        tx, ty, tw, th = t["bbox"]
        t_mid_y = ty + th / 2
        if abs(t_mid_y - p_mid_y) < max(ph, th):
            matching_tess.append(t["text"])
    
    tess_line = " ".join(matching_tess) if matching_tess else "(no match)"
    
    tess_str = f"{tess_line[:48]}"
    paddle_str = f"{p['text'][:48]}"
    print(f"{tess_str:<50} | {paddle_str:<50}")

Tesseract                                          | PaddleOCR                                         
-------------------------------------------------------------------------------------------------------
Treple ted LOc8,                                   | 78C7 pm e dn4S                                    
Guardian Health And Beauty Sdn Phd                 | Guardian Health And Heauty Sdn Bhd                
Jalan Loke Yew Bentong                             | Jalan Loke Yew Bentong                            
90 Ground Floor                                    | 90 Ground Floor                                   
Jalan Bentong                                      | Jalan Bentong                                     
Tel:09-222 6498                                    | Tel:09-222 6498                                   
Company Reg #1101083-T                             | Company Reg #1101083-T                            
GST Reo HOOORS9B748L4                              | GST Reg #00

In [6]:
import importlib
import src.ocr
importlib.reload(src.ocr)

from src.ocr import run_tesseract as tess_imported, run_paddleocr as paddle_imported

tess_check = tess_imported(preprocessed)
paddle_check = paddle_imported(image_path)

print(f"Tesseract:  {len(tess_check)} words (expected {len(tess_results)})")
print(f"PaddleOCR:  {len(paddle_check)} regions (expected {len(paddle_results)})")

Tesseract:  150 words (expected 150)
PaddleOCR:  54 regions (expected 54)
